In [2]:
# =========================================
# Representación con Frames – Caso genérico (LFPDPPP)
# IS-A, herencia, parte-todo, demonios, reglas (forward chaining)
# =========================================

from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Set

# ---------- Infraestructura Frames ----------

class SlotSpec:
    def __init__(
        self,
        name: str,
        default: Any = None,
        allowed: Optional[Set[Any]] = None,
        if_needed: Optional[Callable] = None,
        on_set: Optional[Callable] = None,
    ):
        self.name = name
        self.default = default
        self.allowed = allowed
        self.if_needed = if_needed
        self.on_set = on_set

class FrameType:
    def __init__(self, name: str, parents=None, slots=None):
        self.name = name
        self.parents = parents or []
        self.slots = slots or {}

    def all_slots(self) -> Dict[str, SlotSpec]:
        merged: Dict[str, SlotSpec] = {}
        for p in self.parents:
            merged.update(p.all_slots())
        merged.update(self.slots)
        return merged

class Frame:
    def __init__(self, ftype: "FrameType", name: str, values=None):
        self.ftype = ftype
        self.name = name
        self.values = values or {}
        self.links: Dict[str, List[str]] = {}

    def add_link(self, rel: str, target: str):
        self.links.setdefault(rel, []).append(target)

    def get(self, slot_name: str, kb: "FrameKB"):
        # valor local
        if slot_name in self.values:
            return self.values[slot_name]

        # buscar SlotSpec en herencia
        spec = self.ftype.all_slots().get(slot_name)
        if spec is None:
            return None

        # demonio if-needed
        if spec.if_needed:
            val = spec.if_needed(kb, self)
            self.values[slot_name] = val
            return val

        # default
        return spec.default

    def set(self, slot_name: str, value: Any, kb: "FrameKB"):
        spec = self.ftype.all_slots().get(slot_name)

        # validación allowed
        if spec and spec.allowed and value not in spec.allowed:
            raise ValueError(f"Valor no permitido para {slot_name}: {value}")

        # set
        self.values[slot_name] = value

        # demonio on-set
        if spec and spec.on_set:
            spec.on_set(kb, self, value)

class FrameKB:
    def __init__(self):
        self.types: Dict[str, FrameType] = {}
        self.frames: Dict[str, Frame] = {}

    def add_type(self, t: FrameType):
        self.types[t.name] = t

    def add_frame(self, f: Frame):
        self.frames[f.name] = f

    def get_frame(self, name: str) -> Frame:
        return self.frames[name]

    def is_a(self, subtype: str, supertype: str) -> bool:
        if subtype not in self.types or supertype not in self.types:
            return False

        seen = set()

        def dfs(t: FrameType) -> bool:
            if t.name in seen:
                return False
            seen.add(t.name)
            if t.name == supertype:
                return True
            return any(dfs(p) for p in t.parents)

        return dfs(self.types[subtype])

# ---------- Reglas (forward chaining simple) ----------

@dataclass
class Rule:
    name: str
    condition: Callable
    action: Callable

def infer(kb: FrameKB, rules: List[Rule], frames: List[Frame], max_iter: int = 10):
    for _ in range(max_iter):
        changed_any = False
        for f in frames:
            for r in rules:
                if r.condition(kb, f):
                    changed_any = r.action(kb, f) or changed_any
        if not changed_any:
            break

# ---------- Dominio: Cumplimiento documental LFPDPPP ----------

def build_kb() -> FrameKB:
    kb = FrameKB()

    # Tipos base (8 tipos)
    Entidad = FrameType("Entidad")

    Documento = FrameType("Documento", [Entidad], {
        "nombre": SlotSpec("nombre", default="sin_nombre"),
        "texto": SlotSpec("texto", default=""),
        "origen": SlotSpec("origen", default="interno", allowed={"interno", "externo"}),
    })

    DocumentoNormativo = FrameType("DocumentoNormativo", [Documento], {
        "jurisdiccion": SlotSpec("jurisdiccion", default="MX", allowed={"MX"}),
        "articulos": SlotSpec("articulos", default={}),
    })

    Ley = FrameType("Ley", [DocumentoNormativo], {
        "siglas": SlotSpec("siglas", default="LFPDPPP"),
        "version": SlotSpec("version", default="vigente"),
    })

    DocumentoInterno = FrameType("DocumentoInterno", [Documento], {
        "tipo_documento": SlotSpec(
            "tipo_documento",
            default="aviso_privacidad",
            allowed={"aviso_privacidad", "politica_interna", "contrato_proveedor"},
        ),
        "clausulas": SlotSpec("clausulas", default=[]),
    })

    AvisoPrivacidad = FrameType("AvisoPrivacidad", [DocumentoInterno], {
        "canal": SlotSpec("canal", default="web", allowed={"web", "fisico", "app"}),
    })

    SistemaAnalisis = FrameType("SistemaAnalisis", [Entidad], {
        "nombre_sistema": SlotSpec("nombre_sistema", default="Sistema_Compliance_Frames"),
        "estado_sistema": SlotSpec("estado_sistema", default="activo", allowed={"activo", "pausado"}),
        "contador_alertas": SlotSpec("contador_alertas", default=0),
    })

    # Demonio if-needed: calcula riesgo al consultarse
    def if_needed_nivel_riesgo(kb: FrameKB, h: Frame) -> str:
        frag = (h.get("fragmento", kb) or "").lower()

        menciona_sensibles = (
            "dato sensible" in frag
            or "datos sensibles" in frag
            or "datos personales sensibles" in frag
        )
        menciona_transfer = "transfer" in frag  # transferencia/transferir/transferencias

        # detección simple de ausencia de consentimiento expreso (evita el bug de "contiene la frase")
        ausencia_consent_exp = (
            "sin consentimiento expreso" in frag
            or "no se recaba consentimiento expreso" in frag
            or "no se recabe consentimiento expreso" in frag
            or "no existe consentimiento expreso" in frag
            or "no cuenta con consentimiento expreso" in frag
        )
        menciona_consent_exp = "consentimiento expreso" in frag
        menciona_consentimiento = ("consentimiento" in frag) or ("aceptación" in frag) or ("aceptacion" in frag)

        if menciona_sensibles and (ausencia_consent_exp or not menciona_consent_exp):
            return "alto"

        if menciona_transfer and not menciona_consentimiento:
            return "alto"

        if ("tácito" in frag or "tacito" in frag) and not menciona_sensibles:
            return "medio"

        return "bajo"

    # Demonio on-set: severidad recalcula prioridad y actualiza contador del sistema si es alta
    def on_set_severidad(kb: FrameKB, h: Frame, value: str) -> None:
        mapping = {"bajo": "rutina", "medio": "prioritaria", "alto": "inmediata"}
        h.values["prioridad"] = mapping.get(value, "prioritaria")

        if value == "alto":
            parent = h.links.get("parte_de", [None])[0]
            if parent:
                sysf = kb.get_frame(parent)
                sysf.values["contador_alertas"] = int(sysf.get("contador_alertas", kb)) + 1

    Hallazgo = FrameType("Hallazgo", [Entidad], {
        "fuente": SlotSpec("fuente", default=""),
        "fragmento": SlotSpec("fragmento", default=""),
        "articulo_candidato": SlotSpec("articulo_candidato", default="Art_7_Consentimiento"),
        "nivel_riesgo": SlotSpec("nivel_riesgo", if_needed=if_needed_nivel_riesgo, allowed={"bajo", "medio", "alto"}),
        "severidad": SlotSpec("severidad", default="medio", allowed={"bajo", "medio", "alto"}, on_set=on_set_severidad),
        "prioridad": SlotSpec("prioridad", default="prioritaria"),
        "estado": SlotSpec("estado", default="sin_evaluar", allowed={"sin_evaluar", "en_riesgo", "controlado"}),
        "accion": SlotSpec("accion", default="ninguna"),
    })

    # registrar tipos
    for t in [Entidad, Documento, DocumentoNormativo, Ley, DocumentoInterno, AvisoPrivacidad, SistemaAnalisis, Hallazgo]:
        kb.add_type(t)

    return kb

kb = build_kb()

# ---------- Datos (resumen para modelado) ----------

articulos = {
    "Art_7_Consentimiento": "Consentimiento expreso o tácito; regla general válido el tácito salvo exigencia de expreso.",
    "Art_8_Sensibles": "Datos personales sensibles requieren consentimiento expreso y por escrito.",
    "Art_9_Excepciones": "Supuestos en que no se requiere consentimiento.",
}

frag_consent = (
    "Consentimiento tácito: se entenderá que la persona titular consiente el tratamiento cuando, "
    "habiéndose puesto a su disposición el aviso, no manifieste oposición."
)
frag_sensibles = (
    "Datos personales sensibles: se mencionan datos personales sensibles sin consentimiento expreso."
)
frag_transfer_ok = (
    "Transferencias: el responsable podrá transferir datos; la aceptación del aviso cubre el consentimiento."
)
frag_transfer_sin_base = (
    "Transferencias: se transferirán datos a terceros. No se menciona consentimiento."
)

# ---------- Instancias (10 instancias, 6+ mínimo) ----------

ley = Frame(kb.types["Ley"], "Ley_Ref", {
    "nombre": "LFPDPPP (referencia)",
    "origen": "externo",
    "jurisdiccion": "MX",
    "siglas": "LFPDPPP",
    "version": "vigente",
    "articulos": articulos,
})

aviso = Frame(kb.types["AvisoPrivacidad"], "Aviso_Caso_Banca", {
    "nombre": "Aviso de Privacidad (caso genérico)",
    "origen": "externo",
    "tipo_documento": "aviso_privacidad",
    "canal": "web",
    "clausulas": [frag_consent, frag_sensibles, frag_transfer_ok, frag_transfer_sin_base],
})

sistema = Frame(kb.types["SistemaAnalisis"], "Sistema_1", {
    "nombre_sistema": "Sistema_Compliance_Frames",
    "estado_sistema": "activo",
    "contador_alertas": 0,
})

subsistema = Frame(kb.types["SistemaAnalisis"], "SubSistema_Deteccion", {
    "nombre_sistema": "SubSistema_Deteccion",
    "estado_sistema": "activo",
    "contador_alertas": 0,
})

# Parte-todo (2 niveles) con bidireccionalidad
sistema.add_link("tiene_parte", subsistema.name)
subsistema.add_link("parte_de", sistema.name)

subsistema.add_link("tiene_parte", aviso.name)
aviso.add_link("parte_de", subsistema.name)

# 6 hallazgos
h1 = Frame(kb.types["Hallazgo"], "Hallazgo_1", {"fuente": "Consentimiento", "fragmento": frag_consent})
h2 = Frame(kb.types["Hallazgo"], "Hallazgo_2", {"fuente": "Sensibles", "fragmento": frag_sensibles, "articulo_candidato": "Art_8_Sensibles"})
h3 = Frame(kb.types["Hallazgo"], "Hallazgo_3", {"fuente": "Transferencias_OK", "fragmento": frag_transfer_ok})
h4 = Frame(kb.types["Hallazgo"], "Hallazgo_4", {"fuente": "Transferencias_SinBase", "fragmento": frag_transfer_sin_base})
h5 = Frame(kb.types["Hallazgo"], "Hallazgo_5", {"fuente": "Nota_1", "fragmento": "Revisión general de consistencia del aviso con Art_7_Consentimiento."})
h6 = Frame(kb.types["Hallazgo"], "Hallazgo_6", {"fuente": "Nota_2", "fragmento": "Se mencionan datos personales sensibles y no se recaba consentimiento expreso.", "articulo_candidato": "Art_8_Sensibles"})

for h in [h1, h2, h3, h4, h5, h6]:
    h.add_link("parte_de", sistema.name)

# Registrar en KB
for f in [ley, aviso, sistema, subsistema, h1, h2, h3, h4, h5, h6]:
    kb.add_frame(f)

# ---------- Reglas ----------

def cond_r1(kb: FrameKB, h: Frame) -> bool:
    return (
        h.ftype.name == "Hallazgo"
        and h.get("nivel_riesgo", kb) == "alto"
        and h.get("estado", kb) != "en_riesgo"
    )

def act_r1(kb: FrameKB, h: Frame) -> bool:
    before = h.get("estado", kb)
    h.set("estado", "en_riesgo", kb)
    return before != "en_riesgo"

def cond_r2(kb: FrameKB, h: Frame) -> bool:
    return (
        h.ftype.name == "Hallazgo"
        and h.get("estado", kb) == "en_riesgo"
        and h.get("accion", kb) != "activar_revision_juridica"
    )

def act_r2(kb: FrameKB, h: Frame) -> bool:
    before = h.get("accion", kb)
    h.set("accion", "activar_revision_juridica", kb)
    return before != "activar_revision_juridica"

rules = [
    Rule("R1_marcar_en_riesgo", cond_r1, act_r1),
    Rule("R2_activar_revision", cond_r2, act_r2),
]

# ---------- Pruebas mínimas (salida) ----------

print("PRUEBA 1: is_a")
print("is_a('AvisoPrivacidad','Documento') =", kb.is_a("AvisoPrivacidad", "Documento"))
print("is_a('Ley','DocumentoNormativo') =", kb.is_a("Ley", "DocumentoNormativo"))
print()

print("PRUEBA 2: herencia (slots heredados)")
print("Aviso.origen =", aviso.get("origen", kb))
print("Ley.jurisdiccion =", ley.get("jurisdiccion", kb))
print()

print("PRUEBA 3: agregación (2 niveles)")
print("Sistema_1 tiene_parte ->", sistema.links.get("tiene_parte"))
print("SubSistema_Deteccion parte_de ->", subsistema.links.get("parte_de"))
print("SubSistema_Deteccion tiene_parte ->", subsistema.links.get("tiene_parte"))
print("Aviso parte_de ->", aviso.links.get("parte_de"))
print()

print("PRUEBA 4: demonio if-needed (nivel_riesgo)")
for h in [h1, h2, h3, h4, h5, h6]:
    print(h.name, "| fuente:", h.get("fuente", kb), "| nivel_riesgo:", h.get("nivel_riesgo", kb))
print()

print("PRUEBA 5: demonio on-set (severidad -> prioridad y contador)")
print("contador_alertas antes ->", sistema.get("contador_alertas", kb))
h2.set("severidad", "alto", kb)
print("Hallazgo_2 severidad =", h2.get("severidad", kb), "| prioridad =", h2.get("prioridad", kb))
print("contador_alertas después ->", sistema.get("contador_alertas", kb))
print()

print("PRUEBA 6: infer() (2 reglas)")
antes = [(h.name, h.get("estado", kb), h.get("accion", kb)) for h in [h1, h2, h3, h4, h5, h6]]
print("Antes:", antes)
infer(kb, rules, [h1, h2, h3, h4, h5, h6], max_iter=10)
despues = [(h.name, h.get("estado", kb), h.get("accion", kb)) for h in [h1, h2, h3, h4, h5, h6]]
print("Después:", despues)
print()

print("SALIDA FINAL")
for h in [h1, h2, h3, h4, h5, h6]:
    print(
        h.name,
        "| riesgo:", h.get("nivel_riesgo", kb),
        "| severidad:", h.get("severidad", kb),
        "| prioridad:", h.get("prioridad", kb),
        "| estado:", h.get("estado", kb),
        "| accion:", h.get("accion", kb),
    )

PRUEBA 1: is_a
is_a('AvisoPrivacidad','Documento') = True
is_a('Ley','DocumentoNormativo') = True

PRUEBA 2: herencia (slots heredados)
Aviso.origen = externo
Ley.jurisdiccion = MX

PRUEBA 3: agregación (2 niveles)
Sistema_1 tiene_parte -> ['SubSistema_Deteccion']
SubSistema_Deteccion parte_de -> ['Sistema_1']
SubSistema_Deteccion tiene_parte -> ['Aviso_Caso_Banca']
Aviso parte_de -> ['SubSistema_Deteccion']

PRUEBA 4: demonio if-needed (nivel_riesgo)
Hallazgo_1 | fuente: Consentimiento | nivel_riesgo: medio
Hallazgo_2 | fuente: Sensibles | nivel_riesgo: alto
Hallazgo_3 | fuente: Transferencias_OK | nivel_riesgo: bajo
Hallazgo_4 | fuente: Transferencias_SinBase | nivel_riesgo: bajo
Hallazgo_5 | fuente: Nota_1 | nivel_riesgo: bajo
Hallazgo_6 | fuente: Nota_2 | nivel_riesgo: alto

PRUEBA 5: demonio on-set (severidad -> prioridad y contador)
contador_alertas antes -> 0
Hallazgo_2 severidad = alto | prioridad = inmediata
contador_alertas después -> 1

PRUEBA 6: infer() (2 reglas)
Antes: [(